# 03. Turn a QA dataset into decision-model data

This is the central notebook of the first phase. We take two free public datasets and convert each example into a set of typed questions with supervision.

PubMedQA contains biomedical questions with a yes/no/maybe label and a long answer. Banking77 contains short customer messages with one of 77 intent labels.

The output is JSONL following the schema in `data/SCHEMA.md`. Every code cell below handles one step, and each step ends with a validation.


![QA to decision dataset](../images/qa_to_decision_dataset.png)

*One QA row becomes several typed questions. The JSONL file is the bridge between the data work in this notebook and training and evaluation later.*


## Step 1. Load a small slice of PubMedQA

The loader uses the public Hugging Face dataset when a network is available. If it is not, it falls back to the tiny samples committed in `data/samples`. Either way the exercise is free and reproducible.


In [ ]:
from decision_qa import load_examples
pubmed = load_examples('pubmedqa', split='train', limit=8)
print(len(pubmed))


In [ ]:
first_pm = pubmed[0]
print(first_pm.id)
print(first_pm.label)


In [ ]:
print(first_pm.question)
print(first_pm.answer[:120])


## Step 2. Load a small slice of Banking77

Banking77 is an intent classification dataset. Each row has a short message and a label name such as `card_arrival` or `exchange_rate`.


In [ ]:
banking = load_examples('banking77', split='train', limit=8)
print(len(banking))


In [ ]:
first_bk = banking[0]
print(first_bk.context)
print(first_bk.label)


## Step 3. Set the conversion options

The configuration controls how many distractors we sample, the random seed for reproducibility, and how long a context may be. A fixed seed is important: it makes the generated dataset identical on every run.


In [ ]:
from decision_qa import ConversionConfig
cfg = ConversionConfig(n_distractors=3, seed=42)
print(cfg.seed, cfg.n_distractors)


In [ ]:
assert cfg.n_distractors > 0
print('conversion config OK')


## Step 4. Convert PubMedQA

For each PubMedQA row we emit four questions: two Noul statements (one supported and one contradicted), one Choice over yes/no/maybe, and one Score over the support rubric.


In [ ]:
from decision_qa import convert_pubmedqa
pm_records = convert_pubmedqa(pubmed, cfg, split='train')
print(len(pm_records))


In [ ]:
pm0 = pm_records[0]
print(pm0.id, pm0.split)
print(pm0.state['question'])


In [ ]:
print([q.type for q in pm0.questions])
assert len(pm0.questions) == 4


## Step 5. Inspect the Noul pair

Every record should contain one true and one false statement. This teaches the model both sides of the judgment instead of only the positive case.


In [ ]:
from decision_qa.questions import NoulQuestion
nouls = [q for q in pm0.questions if isinstance(q, NoulQuestion)]
print([q.gold for q in nouls])


In [ ]:
assert sorted(q.gold for q in nouls) == [False, True]
print('Noul pair has both polarities')


In [ ]:
for q in nouls:
    print(q.label, q.statement[:80])


## Step 6. Inspect the Choice question

The three decisions are shuffled. We check that the gold index still points at the original label after shuffling.


In [ ]:
from decision_qa.questions import ChoiceQuestion
pm_choice = next(q for q in pm0.questions if isinstance(q, ChoiceQuestion))
print(pm_choice.options)


In [ ]:
print(pm_choice.options[pm_choice.gold_index])
assert 0 <= pm_choice.gold_index < len(pm_choice.options)


## Step 7. Inspect the Score question

The support rubric is ordered from contradicted to supported. Each record shows either the gold answer or a contradicted one in the rubric, so the gold level alternates between the two ends. This gives the model a real target instead of a constant label.


In [ ]:
from decision_qa.questions import ScoreQuestion
pm_score = next(q for q in pm0.questions if isinstance(q, ScoreQuestion))
print(pm_score.levels)


In [ ]:
print(pm_score.rubric[:120])
print(pm_score.levels[pm_score.gold_index])


In [ ]:
score_golds = [next(q for q in r.questions if isinstance(q, ScoreQuestion)).gold_index for r in pm_records]
print(sorted(set(score_golds)))
assert set(score_golds) == {0, 2}


## Step 8. Convert Banking77

For each message the gold intent is mixed with distractors sampled from the full set of 77 label names, not just the small loaded slice. We use lexical hard negatives so the task is not trivial.


In [ ]:
from decision_qa import convert_banking77
bk_records = convert_banking77(banking, cfg, split='train')
print(len(bk_records))


In [ ]:
bk0 = bk_records[0]
bk_choice = next(q for q in bk0.questions if isinstance(q, ChoiceQuestion))
print(bk_choice.options)


In [ ]:
print(bk_choice.options[bk_choice.gold_index])
assert bk0.state['context'] == first_bk.context


## Step 9. Verify distractors

A distractor must never be the gold label. If it were, the question would have two correct answers and the supervision would be broken.


In [ ]:
gold_label = bk_choice.options[bk_choice.gold_index]
distractors = [o for i, o in enumerate(bk_choice.options) if i != bk_choice.gold_index]
print(distractors)


In [ ]:
assert gold_label not in distractors
print('no duplicate gold among distractors')


## Step 10. Write and read the JSONL

`build_dataset` loads, converts, splits, and writes the file. We then read it back and count the records per split to confirm all three splits are present.


In [ ]:
from decision_qa import build_dataset, read_jsonl
path = build_dataset('pubmedqa', 'data/pubmedqa_decisions.jsonl', limits={'train': 40, 'val': 8, 'test': 8}, cfg=cfg)
print(path.name)


In [ ]:
records = read_jsonl(path)
print(len(records))


In [ ]:
from collections import Counter
counts = Counter(r.split for r in records)
print(dict(counts))
assert set(counts) == {'train', 'val', 'test'}


A quick chart shows the balance of question types. A dataset with only Choice questions would train only one paradigm, so the mix matters.


In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
type_counts = Counter(q.type for record in records for q in record.questions)
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(list(type_counts), list(type_counts.values()), color=['#4C78A8', '#F58518', '#54A24B'])
ax.set_ylabel('Questions')
ax.set_title('Question types in the converted dataset')
plt.show()


## Step 11. Validate the whole dataset

We check two invariants across every record: each Choice gold index is inside its option list, and each Noul statement is non-empty. These checks catch most conversion bugs.


In [ ]:
checked = 0
for record in records:
    for question in record.questions:
        if isinstance(question, ChoiceQuestion):
            assert 0 <= question.gold_index < len(question.options)
        if isinstance(question, NoulQuestion):
            assert question.statement.strip()
        checked += 1
print(checked, 'questions checked')


## Step 12. Preview the request payload

Finally we translate one converted record back into the wire format. This closes the loop: raw QA text goes in, typed questions ready for a decision model come out.


In [ ]:
from decision_qa import to_systemone_payload
import json
preview = to_systemone_payload(records[0].state, records[0].questions)
print(json.dumps(preview, indent=2)[:600])


In [ ]:
assert len(preview['questions']) == 4
print('payload ready for a decision model')


## Next steps

Notebook 04 fine-tunes a small open model on this JSONL. Notebook 05 evaluates the probabilities. The schema is documented in `data/SCHEMA.md`.
